# Inspeção de Qualidade de Peças de Fundição com Visão Computacional

**Mini-Projeto Avaliativo — Módulo 2 (Machine Learning e Visão Computacional)**

Este notebook implementa um pipeline que une **Visão Clássica (OpenCV)** e **Aprendizado Profundo (CNN / TensorFlow-Keras)** para inspecionar automaticamente peças de fundição metálica, classificando-as como **OK** ou **Defeituosa**.

**Etapas:**
1. Análise Exploratória (OpenCV): escala de cinza, blur, limiarização, detecção de bordas e morfologia.
2. Classificação Automatizada (CNN): ingestão em lote, Data Augmentation e treinamento.
3. Auditoria: curvas de Loss e Acurácia (Treino vs. Validação).

In [ ]:
# Bibliotecas de manipulação de dados e visualização
import os
import glob
import zipfile
import numpy as np
import matplotlib.pyplot as plt

# Visão clássica
import cv2

# Deep Learning
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping

# Métricas (bônus)
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

print("TensorFlow:", tf.__version__)
print("OpenCV:", cv2.__version__)

## Sprint 1 — Configuração do Ambiente e Dataset

O dataset é público (*Casting Product Image Data for Quality Inspection*) e **não deve ser versionado no Git** (≈30 MB). Ele fica no Google Drive e é extraído no ambiente do Colab.

In [ ]:
# Monta o Google Drive (somente no Google Colab)
from google.colab import drive
drive.mount('/content/drive')

# Extrai o dataset (zip) do Drive para o ambiente local do Colab
caminho_zip = '/content/drive/MyDrive/casting_line_512x512.zip'
destino = '/content/dataset_pecas'

with zipfile.ZipFile(caminho_zip, 'r') as arquivo_zip:
    arquivo_zip.extractall(destino)

print("Dataset extraído em:", destino)

In [ ]:
# O image_dataset_from_directory espera uma pasta que contenha UMA SUBPASTA POR CLASSE.
# Localizamos automaticamente a pasta que contém as classes 'def_front' e 'ok_front'.
def localizar_pasta_classes(raiz):
    for caminho, subpastas, _ in os.walk(raiz):
        if any(pasta in subpastas for pasta in ('def_front', 'ok_front')):
            return caminho
    raise FileNotFoundError("Não encontrei as pastas de classe 'def_front'/'ok_front'.")

pasta_raiz = localizar_pasta_classes(destino)
print("Pasta de classes:", pasta_raiz)
print("Subpastas:", os.listdir(pasta_raiz))

## Sprint 2 — Análise Exploratória Clássica (OpenCV)

Objetivo: provar visualmente que o defeito (trinca/ranhura) é destacável com processamento clássico.

**Importante:** esta análise é didática e **NÃO é aplicada ao dataset de treino** — servimos para entender o defeito antes de treinar a IA.

In [ ]:
# Seleciona uma amostra de 10 imagens (5 OK e 5 defeituosas)
imagens_ok = sorted(glob.glob(os.path.join(pasta_raiz, 'ok_front', '*')))[:5]
imagens_def = sorted(glob.glob(os.path.join(pasta_raiz, 'def_front', '*')))[:5]
amostra = [('OK', p) for p in imagens_ok] + [('Defeituosa', p) for p in imagens_def]

def carregar_imagem(caminho):
    """Lê a imagem com OpenCV (BGR) e converte para RGB."""
    imagem_bgr = cv2.imread(caminho)
    return cv2.cvtColor(imagem_bgr, cv2.COLOR_BGR2RGB)

print(f"Amostra: {len(imagens_ok)} imagens OK e {len(imagens_def)} defeituosas.")

In [ ]:
# Pipeline clássico — Etapa 1: escala de cinza e suavização de ruído
def pipeline_filtros_basicos(caminho):
    imagem_rgb = carregar_imagem(caminho)
    cinza = cv2.cvtColor(imagem_rgb, cv2.COLOR_RGB2GRAY)
    # Gaussian Blur: média ponderada dos vizinhos (kernel 5x5, sempre ímpar)
    desfoque = cv2.GaussianBlur(cinza, (5, 5), 0)
    return imagem_rgb, cinza, desfoque

# Exibe a amostra completa: Original x Cinza x Blur
fig, eixos = plt.subplots(len(amostra), 3, figsize=(10, 3 * len(amostra)))
for linha, (rotulo, caminho) in enumerate(amostra):
    imagem_rgb, cinza, desfoque = pipeline_filtros_basicos(caminho)
    for eixo, imagem, titulo in zip(
        eixos[linha], [imagem_rgb, cinza, desfoque],
        [f'{rotulo} - Original', 'Escala de cinza', 'Blur (Gaussiano)']
    ):
        eixo.imshow(imagem, cmap='gray')
        eixo.set_title(titulo)
        eixo.axis('off')
plt.tight_layout()
plt.show()

## Sprint 3 — Destaque de Características (Bordas e Morfologia)

Aplicamos limiarização, detecção de bordas (**Canny**) e operações morfológicas (fechamento, erosão e dilatação) para isolar visualmente o defeito da peça.

In [ ]:
# Pipeline clássico completo — Etapa 2: limiarização, bordas e morfologia
def pipeline_completo(caminho):
    imagem_rgb = carregar_imagem(caminho)
    cinza = cv2.cvtColor(imagem_rgb, cv2.COLOR_RGB2GRAY)
    desfoque = cv2.GaussianBlur(cinza, (5, 5), 0)

    # Limiarização: separa objeto (claro) do fundo (escuro)
    _, binaria = cv2.threshold(desfoque, 127, 255, cv2.THRESH_BINARY)

    # Detecção de bordas de Canny (limiar inferior e superior)
    bordas = cv2.Canny(desfoque, 50, 150)

    # Fechamento morfológico: une pequenas descontinuidades das bordas
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    morfologia = cv2.morphologyEx(bordas, cv2.MORPH_CLOSE, kernel)
    return imagem_rgb, cinza, desfoque, binaria, bordas, morfologia

# Plota o pipeline completo para a primeira peça defeituosa
rotulo, caminho = amostra[-1]
imagem_rgb, cinza, desfoque, binaria, bordas, morfologia = pipeline_completo(caminho)

fig, eixos = plt.subplots(1, 5, figsize=(20, 4))
for eixo, imagem, titulo in zip(
    eixos, [imagem_rgb, cinza, desfoque, bordas, morfologia],
    ['Original', 'Escala de cinza', 'Blur', 'Bordas (Canny)', 'Morfologia']
):
    eixo.imshow(imagem, cmap='gray')
    eixo.set_title(titulo)
    eixo.axis('off')
plt.suptitle(f'Pipeline clássico — Peça {rotulo}')
plt.tight_layout()
plt.show()

In [ ]:
# Comparativo: erosão x dilatação x fechamento (kernel 5x5)
kernel5 = np.ones((5, 5), np.uint8)
erosao = cv2.erode(binaria, kernel5, iterations=1)
dilatacao = cv2.dilate(binaria, kernel5, iterations=1)
fechamento = cv2.morphologyEx(binaria, cv2.MORPH_CLOSE, kernel5, iterations=2)

fig, eixos = plt.subplots(1, 4, figsize=(18, 4))
for eixo, imagem, titulo in zip(
    eixos, [binaria, erosao, dilatacao, fechamento],
    ['Binarizada', 'Erosão', 'Dilatação', 'Fechamento']
):
    eixo.imshow(imagem, cmap='gray')
    eixo.set_title(titulo)
    eixo.axis('off')
plt.tight_layout()
plt.show()

### Conclusões da Análise Exploratória

- O **Blur** reduz o ruído industrial (reflexos e granulado do metal), facilitando a detecção.
- O **Canny** evidencia as descontinuidades: a trinca/ranhura aparece como linhas de borda, enquanto a peça OK apresenta contorno mais regular.
- A **morfologia** limpa pequenos ruídos e conecta bordas rompidas.
- Conclusão-chave: *se conseguimos ver o defeito na imagem, a IA também consegue* — por isso a EDA justifica o uso da CNN.